# JAX's foreign function interface

While a wide range of numerical operations can be easily and efficiently implemented using JAX's built in `jax.numpy` and `jax.lax` interfaces, it can sometimes be useful to explicitly call out to external compiled libraries via a "foreign function interface" (FFI).
This can be particularly useful when particular operations have been previously implemented in an optimized C or CUDA library, and it would be non-trivial to reimplement these computations directly using JAX, but it can also be useful for optimizing runtime or memory performance of JAX programs.
That being said, the FFI should typically be considered a last resort option because the XLA compiler that sits in the backend, or the Pallas kernel language, which provides lower level control, typically produce performant code with a lower development and maintenance cost.

JAX's FFI support is provided in two parts:

1. A header-only C++ library from XLA which is packaged as part of JAX as of v0.4.29 or available from the [openxla/xla](https://github.com/openxla/xla) repository, and
2. A Python front end, available in the `jax.extend.ffi` submodule.

In this tutorial we demonstrate the use of both of these components using a simple example, and then go on to discuss some lower-level extensions for more complicated use cases.
We start by presenting the FFI on CPU, and discuss generalizations to GPU or multi-device environments below.

## A simple example

To demonstrate the use of the FFI interface, we will implement a simple "root-mean-square (RMS)" normalization function.
RMS normalization takes an array $x$ with shape $(N,)$ and returns

$$
y_n = \frac{x_n}{\sqrt{\frac{1}{N}\sum_{n=1}^N {x_n}^2 + \epsilon}}
$$

where $\epsilon$ is a tuning parameter used for numerical stability.

This is a somewhat silly example, because it can be easily implemented using JAX as follows:

In [1]:
import jax
import jax.numpy as jnp


def rms_norm_ref(x, eps=1e-5):
  scale = jnp.sqrt(jnp.mean(jnp.square(x), axis=-1, keepdims=True) + eps)
  return x / scale

But, it's just non-trivial enough to be useful for demonstrating some key details of the FFI, while still being straightforward to understand.
We will use this reference implementation to test our FFI version below.

## Backend code

To begin with, we need an implementation of RMS normalization in C++ that we will expose using the FFI.
This isn't meant to be particularly performant, but you could imagine that if you had some new better implementation of RMS normalization in a C++ library, it might have an interface like the following.
So, here's a simple implementation of RMS normalization in C++:

```c++
#include <cmath>
#include <cstdint>

float ComputeRmsNorm(float eps, int64_t size, const float *x, float *y) {
  float sm = 0.0f;
  for (int64_t n = 0; n < size; ++n) {
    sm += x[n] * x[n];
  }
  float scale = 1.0f / std::sqrt(sm / float(size) + eps);
  for (int64_t n = 0; n < size; ++n) {
    y[n] = x[n] * scale;
  }
  return scale;
}
```

and, for our example, this is the function that we want to expose to JAX via the FFI.

### C++ interface

To expose our library function to JAX and XLA, we need to write a thin wrapper using the APIs provided by the header-only library in the [`xla/ffi/api`](https://github.com/openxla/xla/tree/main/xla/ffi/api) directory of the [OpenXLA project](https://github.com/openxla/xla).
For more information about this interface, take a look at [the XLA custom call documentation](https://openxla.org/xla/custom_call), but for our purposes, it's sufficient to know that we can define our implementation as follows:

```c++
#include <utility>

#include "xla/ffi/api/c_api.h"
#include "xla/ffi/api/ffi.h"

namespace ffi = xla::ffi;

std::pair<int64_t, int64_t> GetDims(ffi::Span<const int64_t> dims) {
  if (dims.size() == 0) {
    return std::make_pair(0, 0);
  }
  int64_t lastDim = dims.back();
  int64_t totalSize = 1;
  for (auto n : dims) {
    totalSize *= n;
  }
  return std::make_pair(totalSize, lastDim);
}

ffi::Error RmsNormImpl(float eps, ffi::Buffer<ffi::DataType::F32> x,
                       ffi::Result<ffi::Buffer<ffi::DataType::F32>> y) {
  auto [totalSize, lastDim] = GetDims(x.dimensions);
  if (lastDim == 0) {
    return ffi::Error(ffi::ErrorCode::kInvalidArgument,
                      "RmsNorm input must be an array");
  }
  for (int64_t n = 0; n < totalSize; n += lastDim) {
    ComputeRmsNorm(eps, lastDim, &(x.data[n]), &(y->data[n]));
  }
  return ffi::Error::Success();
}

XLA_FFI_DEFINE_HANDLER(
    RmsNorm, RmsNormImpl,
    ffi::Ffi::Bind()
        .Attr<float>("eps")
        .Arg<ffi::Buffer<ffi::DataType::F32>>(/* x */)
        .Ret<ffi::Buffer<ffi::DataType::F32>>(/* y */));
```

Starting at the bottom, we're using the XLA-provided macro `XLA_FFI_DEFINE_HANDLER` to generate some boilerplate which will expand into a function called `RmsNorm` with the appropriate signature.
But, the important stuff here is all in the call to `ffi::Ffi::Bind()`, where we define the input and output types, and the types of any parameters.

Then, in `RmsNormImpl`, we accept `ffi::Buffer` arguments which include information about the buffer shape, and pointers to the underlying data.
In this implementation, we treat all leading dimensions of the buffer as batch dimensions, and perform RMS normalization over the last axis.
`GetDims` is a helper function providing support for this batching behavior.

### Building and registering an FFI handler

Now that we have our minimal FFI wrapper implemented, we need to expose this function (`RmsNorm`) to Python.
In this tutorial, we use [nanobind](https://nanobind.readthedocs.io) to define a tiny Python module, but it is also possible to compile `RmsNorm` into a shared library and load it using [ctypes](https://docs.python.org/3/library/ctypes.html), a pattern which we discuss below.

For this example, our nanobind module is defined as follows:

```c++
#include <type_traits>

#include "nanobind/nanobind.h"

namespace nb = nanobind;

template <typename T>
nb::capsule EncapsulateFfiCall(T *fn) {
  static_assert(std::is_invocable_r_v<XLA_FFI_Error *, T, XLA_FFI_CallFrame *>,
                "Encapsulated function must be and XLA FFI handler");
  return nb::capsule(reinterpret_cast<void *>(fn), "xla._CUSTOM_CALL_TARGET");
}

NB_MODULE(rms_norm, m) {
  m.def("rms_norm", []() { return EncapsulateFfiCall(RmsNorm); });
}
```

With this in place, we can now compile our library.
Here we use CMake, but you should be able to use your favorite build system without too much trouble.

In [13]:
!cmake -DCMAKE_BUILD_TYPE=Release -B _build .
!cmake --build _build
!cmake --install _build

/Users/danfm/miniforge3/envs/jax-readthedocs/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


-- XLA include directory: /Users/danfm/miniforge3/envs/jax-readthedocs/lib/python3.10/site-packages/jaxlib/include
-- Configuring done (0.7s)
-- Generating done (0.0s)
-- Build files have been written to: /Users/danfm/jax/docs/ffi/_build
[ 84%] Built target nanobind-static
[ 92%] Building CXX object CMakeFiles/rms_norm.dir/rms_norm.cc.o
In file included from /Users/danfm/jax/docs/ffi/rms_norm.cc:23:
/Users/danfm/miniforge3/envs/jax-readthedocs/lib/python3.10/site-packages/jaxlib/include/xla/ffi/api/ffi.h:589:21: warning: explicitly defaulted move assignment operator is implicitly deleted [-Wdefaulted-function-deleted]
  ScratchAllocator& operator=(ScratchAllocator&&) = default;
                    ^
/Users/danfm/miniforge3/envs/jax-readthedocs/lib/python3.10/site-packages/jaxlib/include/xla/ffi/api/ffi.h:602:21: note: move assignment operator of 'ScratchAllocator' is implicitly deleted because field 'diagnostic_' is of reference type 'DiagnosticEngine &'
  DiagnosticEngine& diagnostic_

With this compiled library in hand, we now need to register this handler with XLA via the {func}`jax.lib.xla_client.register_custom_call_target` function.
In our nanobind module above, we implemented a Python function called `rms_norm` which returns a [`PyCapsule`](https://docs.python.org/3/c-api/capsule.html) containing a function pointer to the C++ function `RmsNorm`.
This capsule is the object that we pass to `register_custom_call_target`:

In [3]:
from jax.lib import xla_client
import rms_norm as rms_norm_lib

xla_client.register_custom_call_target(
  "rms_norm", rms_norm_lib.rms_norm(), platform="cpu", api_version=1
)

We note that the `api_version=1` keyword in the call to `register_custom_call_target` is important because it indicates that our function pointer implements this FFI interface.

## Frontend code

Now that we have registered our FFI handler, it is straightforward to call our C++ library from JAX using the {func}`jax.extend.ffi_call` function:

In [4]:
import numpy as np
import jax.extend as jex


def rms_norm(x, eps=1e-5):
  return jex.ffi.ffi_call(
    "rms_norm",
    jax.ShapeDtypeStruct(x.shape, x.dtype),
    x,
    eps=np.float32(eps),  # Note: `numpy` here, not `jax.numpy`
    vectorized=True,
  )


# Test that this gives the same result as our reference implementation
x = jnp.linspace(-0.5, 0.5, 15).reshape((3, 5))
np.testing.assert_allclose(rms_norm(x), rms_norm_ref(x), rtol=1e-5)

There are a few points that we should note here:

- The first argument to `ffi_call` must match the target name that we used when calling `register_custom_call_target` above.
- We need to explicitly cast `eps` to `np.float32` because our FFI library expects a C `float`. Note that we can't use `jax.numpy` here, because these parameters must be static arguments.

### Batching with `vmap`

This implementation supports `vmap` out of the box:

In [5]:
np.testing.assert_allclose(jax.vmap(rms_norm)(x), jax.vmap(rms_norm_ref)(x), rtol=1e-5)

And, since our C++ wrapper directly supports batching over any leading axes, we could set `vectorized=True` in {func}`jax.extend.ffi.ffi_call`.
This means that `vmap` simply passes through to calling our FFI handler directly, which can be seen by inspecting the Jaxpr of `rms_norm`:

In [6]:
jax.make_jaxpr(jax.vmap(rms_norm))(x)

{ lambda ; a:f32[3,5]. let
    b:f32[3,5] = ffi_call[
      eps=1e-05
      ffi_target_name=rms_norm
      result_avals=(ShapedArray(float32[3,5]),)
      vectorized=True
    ] a
  in (b,) }

If `vectorized` is `False` or omitted, `vmap`ping a `ffi_call` will fall back on a {func}`jax.lax.scan` with the `ffi_call` in the body:

In [7]:
def rms_norm_not_vectorized(x, eps=1e-5):
  return jex.ffi.ffi_call(
    "rms_norm",
    jax.ShapeDtypeStruct(x.shape, x.dtype),
    x,
    eps=np.float32(eps),
    vectorized=False,  # This is the default behavior
  )


jax.make_jaxpr(jax.vmap(rms_norm_not_vectorized))(x)

{ lambda ; a:f32[3,5]. let
    b:f32[3,5] = scan[
      _split_transpose=False
      jaxpr={ lambda ; c:f32[5]. let
          d:f32[5] = ffi_call[
            eps=1e-05
            ffi_target_name=rms_norm
            result_avals=(ShapedArray(float32[5]),)
            vectorized=False
          ] c
        in (d,) }
      length=3
      linear=(False,)
      num_carry=0
      num_consts=0
      reverse=False
      unroll=1
    ] a
  in (b,) }

## Differentiation

In [8]:
xla_client.register_custom_call_target(
  "rms_norm_fwd", rms_norm_lib.rms_norm_fwd(), platform="cpu", api_version=1
)
xla_client.register_custom_call_target(
  "rms_norm_bwd", rms_norm_lib.rms_norm_bwd(), platform="cpu", api_version=1
)

In [9]:
def rms_norm_fwd(x, eps=1e-5):
  y, res = jex.ffi.ffi_call(
    "rms_norm_fwd",
    (
      jax.ShapeDtypeStruct(x.shape, x.dtype),
      jax.ShapeDtypeStruct(x.shape[:-1], x.dtype),
    ),
    x,
    eps=np.float32(eps),
    vectorized=True,
  )
  return y, (res, x)


def rms_norm_bwd(eps, res, ct):
  del eps
  res, x = res
  assert res.shape == ct.shape[:-1]
  assert x.shape == ct.shape
  return (
    jex.ffi.ffi_call(
      "rms_norm_bwd",
      jax.ShapeDtypeStruct(ct.shape, ct.dtype),
      res,
      x,
      ct,
      vectorized=True,
    ),
  )


rms_norm = jax.custom_vjp(rms_norm, nondiff_argnums=(1,))
rms_norm.defvjp(rms_norm_fwd, rms_norm_bwd)

In [10]:
_, vjp_fun = jax.vjp(rms_norm, x)

In [11]:
vjp_fun(jnp.ones_like(x))

(Array([[-0.79802966, -0.29913902,  0.19975114,  0.6986419 ,  1.1975322 ],
        [ 9.89465   ,  9.8946495 ,  9.894648  ,  9.894646  ,  9.894645  ],
        [ 1.1975324 ,  0.6986419 ,  0.19975138, -0.29913855, -0.7980287 ]],      dtype=float32),)

In [12]:
jax.vjp(rms_norm_ref, x)[1](jnp.ones_like(x))

(Array([[-0.79802966, -0.29913902,  0.19975114,  0.6986419 ,  1.1975322 ],
        [ 9.89465   ,  9.894649  ,  9.894647  ,  9.894645  ,  9.894643  ],
        [ 1.1975324 ,  0.6986419 ,  0.19975138, -0.29913855, -0.7980287 ]],      dtype=float32),)

## TODO(dfm)

- ctypes + ffi.pycapsule interface
- dtype dispatching
- CUDA
- partitioning
- 